# rl16_grid

> Phase 16 (`specs/multiagent/phase16/`): more training seeds for the incentive result. `RL16_ARMS` (`individual`, exactly Phase 12's v2 individual run at seed 1; `indpen`, Phase 13's arm), `RL16_RUNS` (six new runs, in the interleaved order the Colab batch uses), `RL16_REUSED` (the runs and rows carried over), `rl16Config`, `rl16Suite` (the 15 main boards) and the frozen analysis plan `RL16_PLAN`.

In [ ]:
#| default_exp rl16_grid

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
from explRL.clue_rl import RLConfig
from explRL.coop_grid import MAIN_SEEDS, RL13_ARMS, RL13_PLAN, RL_PLAN, coopHash, evalSuite

RL16_ARMS = dict(individual=dict(scheme='individual'),          # seed 1 = Phase 12's v2 individual run (hash 542eb4f99c29)
                 indpen=RL13_ARMS['indpen'])                      # Phase 13's arm: individual reward, wrong answers priced
RL16_RUNS = (('individual', 2), ('indpen', 3), ('individual', 3), ('indpen', 4), ('individual', 4), ('indpen', 5))  # interleaved
RL16_REUSED = dict(individual_s1=('eval', 'individual'), indpen_s1=('eval_p13', 'indpen_s1'),   # name: (eval dir, model in rows)
                   indpen_s2=('eval_p13', 'indpen_s2'), warm=('eval', 'warm'))
RL16_ALL = dict(individual=['individual_s1', 'individual_s2', 'individual_s3', 'individual_s4'],
                indpen=['indpen_s1', 'indpen_s2', 'indpen_s3', 'indpen_s4', 'indpen_s5'])
RL16_PLAN = dict(
    version=1, frozen=True, base=dict(RL_PLAN=coopHash(RL_PLAN), RL13_PLAN=coopHash(RL13_PLAN)),
    arms=RL16_ARMS, runs=[list(r) for r in RL16_RUNS], allRuns=RL16_ALL,
    seeds='seed s = trainSeed s - 1; adapters <arm>_s<seed>; individual_s1 = Phase 12 v2 individual (config hash 542eb4f99c29)',
    reused={k: list(v) for k, v in RL16_REUSED.items()},
    evaluation='the 15 main boards (N 4, blackboard, 40 turns, seeds 900-904 and 930-939) for the six new adapters; '
               'one Colab batch: training, evaluation, packing; halted runs evaluated from the final adapter saved at the halt',
    metrics=dict(runValue="a run's nan-aware mean of a metric over its 15 main boards",
                 gain="a run's mean over boards of its welfare minus the warm start's on the same board seed",
                 sharing='sharePerHeld (clues shared per clue held), soloShare (share of solves with no shared clue used; '
                         'nan on boards without solves, skipped)'),
    hypotheses=dict(H16_1='permTest(sharePerHeld[indpen], sharePerHeld[individual]) > 0, one-sided exact',
                    H16_2='permTest(soloShare[individual], soloShare[indpen]) > 0, one-sided exact',
                    H16_3='permTest(welfare[indpen], welfare[individual]) > 0, one-sided exact',
                    H16_4='signFlip(gain[indpen]) > 0, one-sided exact'),
    families=dict(primary=['H16_1', 'H16_2', 'H16_3'], secondary=['H16_4']), holm='within the primary family, alpha 0.05',
    intervals='hierarchical bootstrap: runs within each arm, then boards within each run; 2000 draws, seed 0, 95% percentile; '
              'descriptive: it undercovers with 4-5 runs per arm (about 0.90 measured), so the exact tests carry the inference',
    claimRules=dict(sharingTested='H16_1 and H16_2 pass after Holm', scoreContrastTested='H16_3 passes after Holm',
                    beatsImitation='H16_4 p < 0.05 and its hierarchical interval excludes 0; otherwise "seed-dependent"',
                    driftReplicated='at least 3 of the 4 individual runs have a soloShare run value above 0.5'),
    descriptive='each run; drift count; per arm and metric the seed standard deviation against the between-arm difference',
    nboot=2000, ci=0.95, alpha=0.05, exploratory=True)


def rl16Name(arm, seed):
    return f'{arm}_s{seed}'


def rl16Config(arm, seed, model, out):
    "The RLConfig of an RL16 run: RL_PLAN v2's settings with the arm's overrides and trainSeed = seed - 1."
    return RLConfig(model, out, **{**RL_PLAN['rl'], **RL16_ARMS[arm], 'trainSeed': seed - 1})


def rl16Suite(models=None):
    "The main-board evaluation cells (evalSuite's 'main' test, 15 boards) for `models` (default: the six new adapters)."
    models = list(models or [rl16Name(a, s) for a, s in RL16_RUNS])
    cells = [c for c in evalSuite(models) if c['level'].endswith('|main')]
    if sorted({c['seed'] for c in cells}) != sorted(MAIN_SEEDS) or len(MAIN_SEEDS) != 15:
        raise ValueError('the main boards must be the 15 seeds 900-904 and 930-939')
    return cells

## Tests

In [ ]:
import json
from explRL.rl16_grid import *
from explRL.clue_rl import configHash
from explRL.coop_grid import MAIN_SEEDS, coopHash, rl13Config

# AC-2: seed 1 of the individual arm is Phase 12's v2 individual run; indpen equals Phase 13's arm
c12 = rl16Config('individual', 1, '/content/drive/MyDrive/explRL_rl/warm_model', '/content/drive/MyDrive/explRL_rl/runs_v2/individual')
assert configHash(c12) == '542eb4f99c29', configHash(c12)
for s in (1, 2):
    a, b = rl16Config('indpen', s, 'm', 'o'), rl13Config('indpen', s, 'm', 'o')
    assert configHash(a) == configHash(b) and a == b
assert [rl16Config(a, s, 'm', 'o').trainSeed for a, s in RL16_RUNS] == [1, 2, 2, 3, 3, 4]
assert sorted(RL16_RUNS) == [('individual', 2), ('individual', 3), ('individual', 4), ('indpen', 3), ('indpen', 4), ('indpen', 5)]
assert [a for a, _ in RL16_RUNS] == ['individual', 'indpen'] * 3                        # interleaved
# AC-3: 6 x 15 main-board cells, nothing else
cells = rl16Suite()
assert len(cells) == 6 * 15 and all(c['level'].endswith('|main') for c in cells)
assert sorted(MAIN_SEEDS) == [900, 901, 902, 903, 904] + list(range(930, 940))
assert {c['level'].split('|')[0] for c in cells} == {rl16Name(a, s) for a, s in RL16_RUNS}
assert set(RL16_REUSED) == {'individual_s1', 'indpen_s1', 'indpen_s2', 'warm'}
assert sorted(sum(RL16_ALL.values(), [])) == sorted({rl16Name(a, s) for a, s in RL16_RUNS} | {'individual_s1', 'indpen_s1', 'indpen_s2'})
# the plan
assert set(RL16_PLAN['hypotheses']) == {'H16_1', 'H16_2', 'H16_3', 'H16_4'} and RL16_PLAN['families']['primary'] == ['H16_1', 'H16_2', 'H16_3']
json.dumps(RL16_PLAN)
print('RL16_PLAN', coopHash(RL16_PLAN), 'frozen' if RL16_PLAN['frozen'] else 'not frozen')

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()